In [11]:
import time
from typing import TypedDict, Annotated
from operator import add
from langgraph.cache.memory import InMemoryCache
from langgraph.graph import StateGraph,START,END
from langgraph.types import CachePolicy
from loguru import  logger
#1. 声明状态
class OverAllState(TypedDict):
    user:str
    invoke_counts:Annotated[int,add]

#2. 声明节点
def node_a(state:OverAllState) ->OverAllState:
    logger.info("node_a被调用 user:{}",state["user"])
    time.sleep(3) #模拟耗时操作
    logger.info("node_a耗时操作执行完成")
    return {
        "invoke_counts":1
    }

#3. 构建图
builder = StateGraph(state_schema=OverAllState)
builder.add_node(
    "node_a",
    node_a,
    cache_policy=CachePolicy(ttl = 10)
)

builder.add_edge(START,"node_a")
builder.add_edge("node_a",END)

# 4. 编译图
graph = builder.compile(cache= InMemoryCache())

logger.info("首次调用图")
logger.info("运行结果 {}\n\n",graph.invoke({"user":"小明","invoke_counts":0}))
logger.info("相同输入再次调用图")
logger.info("运行结果 {}\n\n",graph.invoke({"user":"小明","invoke_counts":0}))
logger.info("不同的输入调用图")
logger.info("运行结果 {}\n\n",graph.invoke({"user":"小花","invoke_counts":0}))
logger.info("不同的输入再次调用图")
logger.info("运行结果 {}\n\n",graph.invoke({"user":"小花","invoke_counts":0}))

#缓存超时
time.sleep(11)
logger.info("不同的输入再次调用图")
logger.info("运行结果 {}\n\n",graph.invoke({"user":"小花","invoke_counts":0}))


2026-07-06 15:15:13.571 | INFO     | __main__:<module>:36 - 首次调用图
2026-07-06 15:15:13.573 | INFO     | __main__:node_a:15 - node_a被调用 user:小明
2026-07-06 15:15:16.574 | INFO     | __main__:node_a:17 - node_a耗时操作执行完成
2026-07-06 15:15:16.576 | INFO     | __main__:<module>:37 - 运行结果 {'user': '小明', 'invoke_counts': 1}


2026-07-06 15:15:16.576 | INFO     | __main__:<module>:38 - 相同输入再次调用图
2026-07-06 15:15:16.578 | INFO     | __main__:<module>:39 - 运行结果 {'user': '小明', 'invoke_counts': 1}


2026-07-06 15:15:16.578 | INFO     | __main__:<module>:40 - 不同的输入调用图
2026-07-06 15:15:16.579 | INFO     | __main__:node_a:15 - node_a被调用 user:小花
2026-07-06 15:15:19.580 | INFO     | __main__:node_a:17 - node_a耗时操作执行完成
2026-07-06 15:15:19.582 | INFO     | __main__:<module>:41 - 运行结果 {'user': '小花', 'invoke_counts': 1}


2026-07-06 15:15:19.582 | INFO     | __main__:<module>:42 - 不同的输入再次调用图
2026-07-06 15:15:19.584 | INFO     | __main__:<module>:43 - 运行结果 {'user': '小花', 'invoke_counts': 1}


2026-07-06 15:15:3